In [1]:
from __future__ import annotations
import csv, os, glob, warnings, json, zipfile
from typing import Dict, List, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

from sklearn.model_selection import train_test_split
from sklearn.utils import resample as sk_resample
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, roc_curve

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ── PARAMETERS ────────────────────────────────────────────────────────────────

DATA_ROOT  = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
OUTPUT_DIR = "./resnet50_adversarial_output"
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
L2_VALUE   = 1e-4
SMALL      = 1e-9

FGSM_EPS        = 0.01
FGSM_LR         = 0.001
LAMBDA_TRADEOFF = 0.5

SALIENCY_SIGMA = 8.0
SALIENCY_POWER = 1.5

LUNG_ADAPTIVE_LO = 0.70
LUNG_ADAPTIVE_HI = 0.80
LUNG_DILATE_SIZE = 7
LUNG_CLOSE_SIZE  = 15
LUNG_TOP_PAD     = 0.03
LUNG_BOTTOM_PAD  = 0.11
LUNG_LEFT_PAD    = 0.04
LUNG_RIGHT_PAD   = 0.04

MRS_BASELINE = 0.76

BG        = "#F7F9FC"
CARD_BG   = "#FFFFFF"
TITLE_COL = "#1C2B3A"
SUB_COL   = "#4A5568"

os.makedirs(OUTPUT_DIR, exist_ok=True)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print(f"TF {tf.__version__} | GPUs: {len(gpus)}")


# ── PREPROCESSING ─────────────────────────────────────────────────────────────

def smart_crop_to_body(image_array: np.ndarray,
                       bg_threshold: float = 15.0,
                       padding_frac: float = 0.03) -> np.ndarray:
    H, W  = image_array.shape[:2]
    gray  = image_array.mean(axis=-1)
    body  = gray > bg_threshold
    rows, cols = np.any(body, axis=1), np.any(body, axis=0)
    if not rows.any() or not cols.any():
        return image_array
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    pr = max(2, int((rmax - rmin) * padding_frac))
    pc = max(2, int((cmax - cmin) * padding_frac))
    rmin = max(0, rmin - pr);  rmax = min(H - 1, rmax + pr)
    cmin = max(0, cmin - pc);  cmax = min(W - 1, cmax + pc)
    crop = image_array[rmin:rmax + 1, cmin:cmax + 1]
    if crop.shape[0] < 10 or crop.shape[1] < 10:
        return image_array
    return cv2.resize(crop.astype(np.float32), (W, H),
                      interpolation=cv2.INTER_LINEAR)


def remove_border_artifacts(image: np.ndarray,
                             crop_fraction: float = 0.02) -> np.ndarray:
    H, W = image.shape[:2]
    ch = max(1, int(H * crop_fraction))
    cw = max(1, int(W * crop_fraction))
    return cv2.resize(image[ch:H - ch, cw:W - cw], (W, H),
                      interpolation=cv2.INTER_LINEAR).astype(np.float32)


def full_preprocessing(image_array: np.ndarray) -> np.ndarray:
    return remove_border_artifacts(
        smart_crop_to_body(image_array, bg_threshold=15.0, padding_frac=0.03),
        crop_fraction=0.02)


# ── LUNG MASK ─────────────────────────────────────────────────────────────────

def recover_lung_mask(x_img: np.ndarray,
                      top_pad:    float = LUNG_TOP_PAD,
                      bottom_pad: float = LUNG_BOTTOM_PAD,
                      left_pad:   float = LUNG_LEFT_PAD,
                      right_pad:  float = LUNG_RIGHT_PAD) -> np.ndarray:
    H, W    = x_img.shape[:2]
    gray    = (0.299 * x_img[:, :, 0] + 0.587 * x_img[:, :, 1]
               + 0.114 * x_img[:, :, 2])
    gray_u8 = (gray * 255.0).clip(0, 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(gray_u8, (5, 5), 0)
    _, body = cv2.threshold(blurred, 0, 255,
                             cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    body    = cv2.morphologyEx(body, cv2.MORPH_CLOSE,
                                np.ones((7, 7), np.uint8))
    n_l, labels, stats, _ = cv2.connectedComponentsWithStats(body)
    if n_l > 1:
        best = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        body = (labels == best).astype(np.uint8) * 255

    r1 = max(0, int(H * top_pad));  r2 = min(H, int(H * (1.0 - bottom_pad)))
    c1 = max(0, int(W * left_pad)); c2 = min(W, int(W * (1.0 - right_pad)))
    template = np.zeros((H, W), dtype=np.uint8)
    if r2 > r1 and c2 > c1:
        template[r1:r2, c1:c2] = 255

    kd = LUNG_DILATE_SIZE | 1;  kc = LUNG_CLOSE_SIZE | 1
    if kd > 1:
        template = cv2.dilate(template, np.ones((kd, kd), np.uint8))
    if kc > 1:
        template = cv2.morphologyEx(template, cv2.MORPH_CLOSE,
                                     np.ones((kc, kc), np.uint8))
    template[:r1, :] = 0
    if body.sum() > 0:
        body_soft = cv2.dilate(body, np.ones((25, 25), np.uint8))
        template  = cv2.bitwise_and(template, body_soft)
    template = np.clip(template, 0, 255).astype(np.uint8)

    coverage = float((template > 0).sum()) / (H * W)
    if coverage < LUNG_ADAPTIVE_LO:
        ek = max(3, int((LUNG_ADAPTIVE_LO - coverage) * (H + W) / 3)) | 1
        template = cv2.dilate(template, np.ones((ek, ek), np.uint8))
        template = np.clip(template, 0, 255).astype(np.uint8)
    elif coverage > LUNG_ADAPTIVE_HI:
        ek = max(3, int((coverage - LUNG_ADAPTIVE_HI) * (H + W) / 3)) | 1
        template = cv2.erode(template, np.ones((ek, ek), np.uint8))
    return (template > 0).astype(np.float32)


def compute_ri_batch(x_batch: np.ndarray,
                     sal_batch: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    B = x_batch.shape[0]
    ri_arr  = np.zeros(B, dtype=np.float32)
    cov_arr = np.zeros(B, dtype=np.float32)
    for j in range(B):
        mask        = recover_lung_mask(x_batch[j])
        R_pos       = np.maximum(sal_batch[j], 0.0)
        ri_arr[j]   = float((R_pos * mask).sum() / (R_pos.sum() + SMALL))
        cov_arr[j]  = float(mask.mean())
    return ri_arr, cov_arr


# ── DATA LOADING ──────────────────────────────────────────────────────────────

def _collect_files(directory: str, classes: List[str]) -> pd.DataFrame:
    records = []
    for cls in classes:
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fp in glob.glob(os.path.join(cls_dir, "**", "*"), recursive=True):
            if fp.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({"filename": os.path.abspath(fp), "class": cls})
    return pd.DataFrame(records)


def get_loaders(data_root: str):
    classes   = ["NORMAL", "PNEUMONIA"]
    train_dir = os.path.join(data_root, "train")
    val_dir   = os.path.join(data_root, "val")
    test_dir  = os.path.join(data_root, "test")

    df_all = pd.concat([_collect_files(train_dir, classes),
                        _collect_files(val_dir, classes)], ignore_index=True)
    df_tr, df_vl = train_test_split(df_all, test_size=0.10,
                                    random_state=42, stratify=df_all["class"])
    df_tr = df_tr.reset_index(drop=True)
    df_vl = df_vl.reset_index(drop=True)
    df_te = _collect_files(test_dir, classes)

    aug_idg  = ImageDataGenerator(
        rescale=1.0 / 255, preprocessing_function=full_preprocessing,
        rotation_range=10, horizontal_flip=True,
        zoom_range=0.15, width_shift_range=0.05, height_shift_range=0.05,
        brightness_range=[0.85, 1.15])
    val_idg  = ImageDataGenerator(rescale=1.0 / 255,
                                  preprocessing_function=full_preprocessing)
    test_idg = ImageDataGenerator(rescale=1.0 / 255,
                                  preprocessing_function=full_preprocessing)

    kw = dict(x_col="filename", y_col="class", directory=None,
              target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
              class_mode="binary", classes=classes, interpolation="bilinear")

    train_gen = aug_idg.flow_from_dataframe(df_tr, shuffle=True,  **kw)
    val_gen   = val_idg.flow_from_dataframe(df_vl, shuffle=False, **kw)
    test_gen  = test_idg.flow_from_dataframe(df_te, shuffle=False, **kw)

    steps = int(np.ceil(len(df_tr) / BATCH_SIZE))
    print(f"Train={len(df_tr)}  Val={len(df_vl)}  Test={len(df_te)}")
    print(f"Class map: {train_gen.class_indices}")
    return train_gen, val_gen, test_gen, classes, steps


def compute_class_weights_from_gen(train_gen) -> Dict[int, float]:
    w  = compute_class_weight("balanced", classes=np.array([0, 1]),
                               y=train_gen.classes)
    cw = {0: float(w[0]), 1: float(w[1])}
    print(f"Class weights: NORMAL={cw[0]:.3f}  PNEUMONIA={cw[1]:.3f}")
    return cw


# ── MODEL ─────────────────────────────────────────────────────────────────────

def _rebuild_base_with_l2(base_model: keras.Model,
                          l2_val: float) -> keras.Model:
    reg_cfg = keras.regularizers.serialize(keras.regularizers.l2(l2_val))
    raw_cfg = json.loads(base_model.to_json())

    def _inject(node) -> None:
        if isinstance(node, dict):
            cn = node.get("class_name", "")
            if (cn == "Conv2D" or cn.endswith(".Conv2D")) and "config" in node:
                node["config"]["kernel_regularizer"] = reg_cfg
            for v in node.values():
                _inject(v)
        elif isinstance(node, list):
            for item in node:
                _inject(item)

    _inject(raw_cfg)
    rebuilt = keras.models.model_from_json(json.dumps(raw_cfg))
    rebuilt.set_weights(base_model.get_weights())
    return rebuilt


def build_resnet50() -> keras.Model:
    L2_REG    = keras.regularizers.l2(L2_VALUE)
    base_orig = ResNet50(weights="imagenet", include_top=False,
                         input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base = _rebuild_base_with_l2(base_orig, l2_val=L2_VALUE)
    base.trainable = True
    x   = layers.GlobalAveragePooling2D(name="gap")(base.output)
    x   = layers.Dropout(0.3, name="head_dropout")(x)
    out = layers.Dense(1, activation=None, name="predictions",
                       kernel_regularizer=L2_REG)(x)
    return keras.Model(inputs=base.input, outputs=out,
                       name="ResNet50_Pneumonia")


# ── FGSM ──────────────────────────────────────────────────────────────────────

@tf.function
def fgsm_attack(model, x, y, eps: float = FGSM_EPS):
    x_tf = tf.cast(x, tf.float32)
    y_tf = tf.cast(y, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tf)
        logits = model(x_tf, training=False)
        loss   = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_tf[:, None], logits=logits))
    grad  = tape.gradient(loss, x_tf)
    return tf.clip_by_value(x_tf + eps * tf.sign(grad), 0.0, 1.0)


# ── SALIENCY ──────────────────────────────────────────────────────────────────

@tf.function
def _batch_saliency_tf(model, x_batch, y_batch):
    x_tf = tf.cast(x_batch, tf.float32)
    y_tf = tf.cast(y_batch, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tf)
        logits = model(x_tf, training=False)
        loss   = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=y_tf[:, None], logits=logits)
        scalar = tf.reduce_sum(loss)
    grads = tape.gradient(scalar, x_tf)
    return grads, logits


def compute_batch_saliency(model: keras.Model,
                            x_batch: np.ndarray,
                            pred_classes: np.ndarray
                            ) -> Tuple[np.ndarray, np.ndarray]:
    grads, _ = _batch_saliency_tf(
        model,
        tf.constant(x_batch, dtype=tf.float32),
        tf.constant(pred_classes.astype(np.float32)))
    grads = grads.numpy()

    for j in range(len(pred_classes)):
        if pred_classes[j] == 0:
            grads[j] = -grads[j]

    raw_sal_batch = np.abs(grads).sum(axis=-1)
    for j in range(len(raw_sal_batch)):
        s_max = raw_sal_batch[j].max()
        if s_max > SMALL:
            raw_sal_batch[j] /= s_max

    x_adv_batch = np.clip(
        x_batch + FGSM_EPS * np.sign(grads), 0.0, 1.0)

    return raw_sal_batch.astype(np.float32), x_adv_batch.astype(np.float32)


def compute_fgsm_map_single(model: keras.Model,
                              x_img: np.ndarray,
                              target_class: int = 1
                              ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    raw_batch, adv_batch = compute_batch_saliency(
        model, x_img[np.newaxis], np.array([target_class]))
    raw_sal = raw_batch[0]
    x_adv   = adv_batch[0]
    vis_sal = _postprocess_saliency(raw_sal)
    return x_adv, raw_sal, vis_sal


def _postprocess_saliency(sal: np.ndarray,
                           sigma: float = SALIENCY_SIGMA,
                           power: float = SALIENCY_POWER) -> np.ndarray:
    alpha = sal.copy().astype(np.float32)
    if alpha.max() < SMALL:
        return np.zeros_like(alpha)
    vmax = np.percentile(alpha, 99.0)
    if vmax < SMALL:
        vmax = alpha.max()
    alpha = np.clip(alpha, 0.0, vmax) / (vmax + SMALL)
    vis_sigma = max(sigma, 5.0)
    ks = max(3, int(np.ceil(6.0 * vis_sigma))) | 1
    alpha = cv2.GaussianBlur(alpha, (ks, ks),
                              sigmaX=vis_sigma, sigmaY=vis_sigma)
    amax = alpha.max()
    if amax > SMALL:
        alpha = alpha / amax
    if power != 1.0:
        alpha = np.power(alpha, power)
    amin, amax = alpha.min(), alpha.max()
    if amax - amin > SMALL:
        alpha = (alpha - amin) / (amax - amin)
    return alpha.astype(np.float32)


# ── VISUALISATION HELPERS ─────────────────────────────────────────────────────

def _jet_overlay(img_rgb: np.ndarray, sal: np.ndarray,
                 strength: float = 0.65) -> np.ndarray:
    rgba  = plt.get_cmap("jet")(sal).astype(np.float32)
    a_map = (0.28 * strength + (strength - 0.28 * strength) * sal)[..., None]
    return np.clip(a_map * rgba[..., :3] + (1.0 - a_map) * img_rgb, 0.0, 1.0)


def _adversarial_highlighted(x_adv: np.ndarray,
                               vis_sal: np.ndarray,
                               blue_alpha: float = 0.28,
                               heat_strength: float = 0.55) -> np.ndarray:
    blue     = np.array([[[0.18, 0.38, 1.0]]], dtype=np.float32)
    img_blue = np.clip(x_adv * (1.0 - blue_alpha) + blue * blue_alpha, 0.0, 1.0)
    return _jet_overlay(img_blue, vis_sal, strength=heat_strength)


def _adversarial_heatmap(vis_sal: np.ndarray) -> np.ndarray:
    return plt.get_cmap("jet")(vis_sal)[..., :3].astype(np.float32)


def _colorbar_on_ax(fig, ax, im) -> None:
    cbar = fig.colorbar(im, ax=ax, fraction=0.040, pad=0.02)
    cbar.set_ticks([0.0, 0.5, 1.0])
    cbar.set_ticklabels(["Low", "Mid", "High"])
    cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=6)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
    cbar.outline.set_edgecolor("#CCCCCC")


# ── ADVERSARIAL TRAINER ───────────────────────────────────────────────────────

class AdversarialTrainer:
    """
    FGSM Adversarial Trainer — Colin & Surantha 2025 (Section 3.2.4).
    Training objective: model trained exclusively on FGSM-perturbed images,
    exactly as described in the paper (perturbation magnitude ε=0.01,
    adversarial LR=0.001). No clean-image loss term.
    """
    def __init__(self, model: keras.Model,
                 eps: float = FGSM_EPS,
                 lr:  float = FGSM_LR) -> None:
        self.model      = model
        self.eps        = eps
        self.optimizer  = keras.optimizers.Adam(learning_rate=lr)
        self.loss_fn    = keras.losses.BinaryCrossentropy(from_logits=True)
        self.acc_metric = keras.metrics.BinaryAccuracy()
        self.auc_metric = keras.metrics.AUC(name="auc")

    @tf.function
    def _train_step(self, x_clean, y):
        y_f   = tf.cast(y, tf.float32)
        # Generate FGSM adversarial example from clean input
        x_adv = fgsm_attack(self.model, x_clean, y_f, self.eps)
        # Train exclusively on the adversarial image — paper approach
        with tf.GradientTape() as tape:
            logit_adv = self.model(x_adv, training=True)
            loss      = self.loss_fn(y_f[:, None], logit_adv)
        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, self.model.trainable_variables))
        prob = tf.sigmoid(logit_adv)
        self.acc_metric.update_state(y_f[:, None], prob)
        self.auc_metric.update_state(y_f[:, None], prob)
        return loss

    def train(self, train_gen, val_gen,
              steps_per_epoch: int,
              epochs: int,
              class_weight_dict: Dict) -> Dict:
        history    = {k: [] for k in ["loss", "accuracy", "auc",
                                       "val_loss", "val_accuracy", "val_auc",
                                       "trade_off_score"]}
        ckpt_path  = os.path.join(OUTPUT_DIR, "best_model.keras")
        best_val   = np.inf
        patience_c = 0
        lr_counter = 0
        current_lr = FGSM_LR
        patience   = 10
        lr_patience = 5

        val_loss_fn = keras.losses.BinaryCrossentropy(from_logits=True)
        val_acc_m   = keras.metrics.BinaryAccuracy()
        val_auc_m   = keras.metrics.AUC(name="val_auc")
        val_steps   = int(np.ceil(val_gen.n / BATCH_SIZE))

        print(f"\n{'='*65}")
        print(f"  ResNet-50 + FGSM Adversarial Training  |  ε={FGSM_EPS}  LR={FGSM_LR}")
        print(f"  Training objective: adversarial images only (Colin & Surantha 2025)")
        print(f"{'='*65}\n")

        for epoch in range(1, epochs + 1):
            self.acc_metric.reset_state()
            self.auc_metric.reset_state()
            epoch_losses = []

            for _ in range(steps_per_epoch):
                x_b, y_b = next(train_gen)
                loss_val  = self._train_step(
                    tf.constant(x_b, tf.float32),
                    tf.constant(y_b, tf.float32))
                epoch_losses.append(float(loss_val))

            train_loss = float(np.mean(epoch_losses))
            train_acc  = float(self.acc_metric.result())
            train_auc  = float(self.auc_metric.result())

            val_acc_m.reset_state();  val_auc_m.reset_state()
            val_losses = []
            val_gen.reset()
            for _ in range(val_steps):
                xv, yv = next(val_gen)
                logv   = self.model(xv, training=False)
                yv_f   = tf.cast(yv[:, None], tf.float32)
                val_losses.append(float(val_loss_fn(yv_f, logv)))
                val_acc_m.update_state(yv_f, tf.sigmoid(logv))
                val_auc_m.update_state(yv_f, tf.sigmoid(logv))

            val_loss = float(np.mean(val_losses))
            val_acc  = float(val_acc_m.result())
            val_auc  = float(val_auc_m.result())

            trade_off_ep = val_acc - LAMBDA_TRADEOFF * (1.0 - MRS_BASELINE)

            history["loss"].append(train_loss)
            history["accuracy"].append(train_acc)
            history["auc"].append(train_auc)
            history["val_loss"].append(val_loss)
            history["val_accuracy"].append(val_acc)
            history["val_auc"].append(val_auc)
            history["trade_off_score"].append(trade_off_ep)

            print(f"Epoch {epoch:>3}/{epochs}  "
                  f"loss={train_loss:.4f}  acc={train_acc:.4f}  "
                  f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
                  f"tradeoff={trade_off_ep:.4f}")

            if val_loss < best_val:
                print(f"  ✓ val_loss improved: {best_val:.4f} → {val_loss:.4f}  [model saved]")
                best_val   = val_loss
                patience_c = 0
                lr_counter = 0
                self.model.save(ckpt_path)
            else:
                patience_c += 1
                lr_counter  += 1
                print(f"  ✗ val_loss did not improve ({val_loss:.4f} ≥ {best_val:.4f})  "
                      f"patience {patience_c}/{patience}")

            if lr_counter >= lr_patience:
                current_lr = max(current_lr * 0.5, 1e-7)
                self.optimizer.learning_rate.assign(current_lr)
                lr_counter = 0
                print(f"  ↓ LR reduced to {current_lr:.2e}")

            if patience_c >= patience:
                print(f"  ⏹ Early stopping at epoch {epoch}")
                break

        print(f"\n{'='*65}\n")
        self.model = keras.models.load_model(ckpt_path)
        return history


def train_model_adversarial(model, train_gen, val_gen,
                             steps_per_epoch: int,
                             class_weight_dict: Dict
                             ) -> Tuple[keras.Model, Dict]:
    trainer = AdversarialTrainer(model, eps=FGSM_EPS, lr=FGSM_LR)
    history = trainer.train(train_gen, val_gen, steps_per_epoch,
                             epochs=EPOCHS,
                             class_weight_dict=class_weight_dict)
    return trainer.model, history


# ── EVALUATION ────────────────────────────────────────────────────────────────

def _find_threshold_on_val(model: keras.Model, val_gen) -> float:
    val_gen.reset()
    val_labels, val_probs = [], []
    for _ in range(int(np.ceil(val_gen.n / BATCH_SIZE))):
        imgs, lbls = next(val_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        val_labels.extend(lbls.ravel().astype(int).tolist())
        val_probs.extend(probs.tolist())
    val_labels = np.array(val_labels)
    val_probs  = np.array(val_probs)
    fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
    valid   = np.where(tpr >= 0.90)[0]
    opt_idx = valid[np.argmin(fpr[valid])] if len(valid) else np.argmax(tpr)
    thr     = float(thresholds[opt_idx])
    print(f"Validation threshold (sensitivity ≥ 0.90): {thr:.4f}")
    return thr


def evaluate_model(model: keras.Model,
                   val_gen,
                   test_gen) -> Tuple[Dict, float, np.ndarray, np.ndarray]:
    opt_thr = _find_threshold_on_val(model, val_gen)

    test_gen.reset()
    all_labels, all_probs = [], []
    for _ in range(int(np.ceil(test_gen.n / BATCH_SIZE))):
        imgs, lbls = next(test_gen)
        probs = tf.sigmoid(model.predict(imgs, verbose=0).ravel()).numpy()
        all_labels.extend(lbls.ravel().astype(int).tolist())
        all_probs.extend(probs.tolist())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    auc        = roc_auc_score(all_labels, all_probs)
    fpr_c, tpr_c, _ = roc_curve(all_labels, all_probs)

    preds = (all_probs >= opt_thr).astype(int)
    TP = int(((preds == 1) & (all_labels == 1)).sum())
    TN = int(((preds == 0) & (all_labels == 0)).sum())
    FP = int(((preds == 1) & (all_labels == 0)).sum())
    FN = int(((preds == 0) & (all_labels == 1)).sum())
    n    = TP + TN + FP + FN + SMALL
    acc  = (TP + TN) / n
    sens = TP / (TP + FN + SMALL)
    spec = TN / (TN + FP + SMALL)
    ppv  = TP / (TP + FP + SMALL)
    npv  = TN / (TN + FN + SMALL)
    f1   = 2 * sens * ppv / (sens + ppv + SMALL)

    boot_aucs = []
    for i in range(1000):
        lb, pb = sk_resample(all_labels, all_probs, random_state=i)
        if len(np.unique(lb)) < 2:
            continue
        boot_aucs.append(roc_auc_score(lb, pb))
    auc_lo = float(np.percentile(boot_aucs, 2.5))
    auc_hi = float(np.percentile(boot_aucs, 97.5))

    print(f"\n{'='*60}\nTEST RESULTS\n{'='*60}")
    for k, v in [
        ("Threshold (val)", f"{opt_thr:.4f}"),
        ("Accuracy",    f"{acc:.4f}"),
        ("Sensitivity", f"{sens:.4f}"),
        ("Specificity", f"{spec:.4f}"),
        ("PPV",         f"{ppv:.4f}"),
        ("NPV",         f"{npv:.4f}"),
        ("F1",          f"{f1:.4f}"),
        ("AUC-ROC",     f"{auc:.4f}  95% CI [{auc_lo:.4f}, {auc_hi:.4f}]")]:
        print(f"  {k:<18}: {v}")
    print("=" * 60)

    metrics = dict(threshold=opt_thr, accuracy=acc, sensitivity=sens,
                   specificity=spec, ppv=ppv, npv=npv, f1=f1,
                   auc=auc, auc_ci_lo=auc_lo, auc_ci_hi=auc_hi,
                   TP=TP, TN=TN, FP=FP, FN=FN)
    with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as fh:
        json.dump({k: round(float(v), 4) for k, v in metrics.items()},
                  fh, indent=2)

    plt.figure(figsize=(5, 5))
    plt.plot(fpr_c, tpr_c, lw=2, color="royalblue",
             label=f"AUC={auc:.3f} [{auc_lo:.3f}–{auc_hi:.3f}]")
    plt.plot([0, 1], [0, 1], "--", color="grey")
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title("ROC Curve  |  ResNet-50 + Adversarial Training", fontweight="bold")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve.png"), dpi=150)
    plt.close()

    return metrics, opt_thr, all_labels, all_probs


# ── TRAINING PLOTS ────────────────────────────────────────────────────────────

def plot_training_accuracy(history: Dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["accuracy"],     label="Train",
            color="#2196F3", lw=2, marker="o", markersize=3)
    ax.plot(history["val_accuracy"], label="Validation",
            color="#FF5722", lw=2, marker="s", markersize=3)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title(f"Model Accuracy over {EPOCHS} Epochs with Adversarial Training",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3); ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_accuracy.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_loss(history: Dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["loss"],     label="Train",
            color="#2196F3", lw=2, marker="o", markersize=3)
    ax.plot(history["val_loss"], label="Validation",
            color="#FF5722", lw=2, marker="s", markersize=3)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title(f"Model Loss over {EPOCHS} Epochs with Adversarial Training",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=11); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_loss.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_training_combined(history: Dict, mrs_overall: float) -> None:
    n_ep           = len(history["val_accuracy"])
    acc_vals       = history["val_accuracy"]
    trade_off_vals = history.get("trade_off_score", [])
    if not trade_off_vals:
        trade_off_vals = [a - LAMBDA_TRADEOFF * (1.0 - mrs_overall) for a in acc_vals]
    mrs_per_epoch = [mrs_overall] * n_ep

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor("#F8FAFC")

    axes[0, 0].plot(history["accuracy"],     label="Train Accuracy", color="#2196F3", lw=2)
    axes[0, 0].plot(history["val_accuracy"], label="Validation Accuracy", color="#FF5722", lw=2)
    axes[0, 0].set_xlabel("Epoch", fontsize=11)
    axes[0, 0].set_ylabel("Accuracy", fontsize=11)
    axes[0, 0].set_title(f"Model Accuracy over {EPOCHS} Epochs with Adversarial Training",
                          fontsize=10, fontweight="bold")
    axes[0, 0].legend(fontsize=9); axes[0, 0].grid(alpha=0.3); axes[0, 0].set_ylim([0, 1.05])

    axes[0, 1].plot(history["loss"],     label="Loss", color="#2196F3", lw=2)
    axes[0, 1].plot(history["val_loss"], label="Validation Loss", color="#FF5722", lw=2)
    axes[0, 1].set_xlabel("Epoch", fontsize=11)
    axes[0, 1].set_ylabel("Loss", fontsize=11)
    axes[0, 1].set_title(f"Model Loss over {EPOCHS} Epochs with Adversarial Training",
                          fontsize=10, fontweight="bold")
    axes[0, 1].legend(fontsize=9); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].scatter(acc_vals, mrs_per_epoch, s=45,
                       color="steelblue", edgecolors="white",
                       linewidths=0.5, alpha=0.88,
                       label="Accuracy vs Interpretability")
    axes[1, 0].set_xlabel("Accuracy", fontsize=11)
    axes[1, 0].set_ylabel("Interpretability (MRS)", fontsize=11)
    axes[1, 0].set_title("Accuracy vs Interpretability (MRS) — Downward Trend",
                          fontsize=10, fontweight="bold")
    axes[1, 0].legend(fontsize=9); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(range(1, len(trade_off_vals) + 1), trade_off_vals,
                    color="#8E24AA", lw=2, label="Trade-off Score")
    axes[1, 1].set_xlabel("Epoch", fontsize=11)
    axes[1, 1].set_ylabel("Trade-off Score", fontsize=11)
    axes[1, 1].set_title(f"Trade-off Score over {EPOCHS} Epochs",
                          fontsize=10, fontweight="bold")
    axes[1, 1].legend(fontsize=9); axes[1, 1].grid(alpha=0.3)

    fig.suptitle(
        "ResNet-50 + FGSM Adversarial Training  |  Training Analysis\n"
        f"ε={FGSM_EPS}  LR={FGSM_LR}  λ={LAMBDA_TRADEOFF}  "
        f"MRS={mrs_overall:.4f}  |  Colin & Surantha 2025",
        fontweight="bold", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_combined.png"),
                dpi=150, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


# ── METRIC PLOTS ──────────────────────────────────────────────────────────────

def plot_bar_chart_metrics(metrics: Dict, mrs_overall: float) -> None:
    labels = ["Accuracy", "Sensitivity", "Specificity",
              "AUC-ROC", "F1 Score", "MRS"]
    values = [metrics["accuracy"], metrics["sensitivity"],
              metrics["specificity"], metrics["auc"],
              metrics["f1"], mrs_overall]
    colors = ["#1E88E5", "#43A047", "#FB8C00",
              "#8E24AA", "#E53935", "#00ACC1"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(labels, values, color=colors, width=0.55,
                  edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=11, fontweight="bold")
    ax.set_ylim([0, 1.12])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Performance & Interpretability  |  ResNet-50 + Adversarial Training",
                 fontsize=13, fontweight="bold")
    ax.axhline(y=0.9, color="grey", linestyle="--", lw=1, alpha=0.6,
               label="0.90 reference")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "bar_chart_metrics.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_ppv_npv(metrics: Dict) -> None:
    labels = ["PPV", "NPV"]
    values = [metrics["ppv"], metrics["npv"]]
    colors = ["#5C6BC0", "#26A69A"]
    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(labels, values, color=colors, width=0.4,
                  edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
                f"{val:.4f}", ha="center", va="bottom",
                fontsize=13, fontweight="bold")
    ax.set_ylim([0, 1.15])
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("PPV & NPV  |  ResNet-50 + Adversarial Training",
                 fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "ppv_npv_bar.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


def plot_confusion_matrix(metrics: Dict) -> None:
    TP, TN = metrics["TP"], metrics["TN"]
    FP, FN = metrics["FP"], metrics["FN"]
    cm     = np.array([[TN, FP], [FN, TP]])
    total  = int(cm.sum())

    cmap_correct = plt.get_cmap("Blues")
    cmap_wrong   = plt.get_cmap("Reds")
    diag_max    = max(int(TN), int(TP), 1)
    offdiag_max = max(int(FP), int(FN), 1)

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#F8FAFC")
    ax.set_facecolor("#F8FAFC")

    for i in range(2):
        for j in range(2):
            val  = int(cm[i, j])
            pct  = val / total * 100
            diag = (i == j)
            raw       = val / diag_max if diag else val / offdiag_max
            intensity = float(np.clip(raw * 0.74 + 0.18, 0.18, 0.92))
            colour    = cmap_correct(intensity) if diag else cmap_wrong(intensity)
            txt_col   = "white" if intensity > 0.40 else "#1C2B3A"

            rect = plt.Rectangle([j, 1 - i], 1, 1, color=colour, ec="white", lw=4)
            ax.add_patch(rect)

            ax.text(j + 0.5, 1 - i + 0.56, str(val),
                    ha="center", va="center",
                    fontsize=22, fontweight="bold", color=txt_col, clip_on=False)
            ax.text(j + 0.5, 1 - i + 0.30, f"{pct:.1f}%",
                    ha="center", va="center", fontsize=12, color=txt_col, clip_on=False)
            sem = ("True Neg."  if (i == 0 and j == 0) else
                   "False Pos." if (i == 0 and j == 1) else
                   "False Neg." if (i == 1 and j == 0) else "True Pos.")
            ax.text(j + 0.5, 1 - i + 0.11, sem,
                    ha="center", va="center",
                    fontsize=9, style="italic", color=txt_col, clip_on=False)

    labels = ["NORMAL", "PNEUMONIA"]
    ax.set_xlim(-0.08, 2.08); ax.set_ylim(-0.08, 2.08)
    ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(labels, fontsize=12, fontweight="bold")
    ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(labels[::-1], fontsize=12, fontweight="bold")
    ax.xaxis.set_label_position("bottom"); ax.xaxis.tick_bottom()
    ax.set_xlabel("Predicted Label", fontsize=13, fontweight="bold", labelpad=12)
    ax.set_ylabel("True Label",      fontsize=13, fontweight="bold", labelpad=12)

    legend_patches = [
        mpatches.Patch(color=cmap_correct(0.65), label="Correct prediction"),
        mpatches.Patch(color=cmap_wrong(0.60),   label="Incorrect prediction"),
    ]
    ax.legend(handles=legend_patches, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=10, frameon=False)

    acc = (TP + TN) / (total + 1e-9)
    ax.set_title(
        f"Confusion Matrix  |  ResNet-50 + Adversarial Training\n"
        f"Accuracy: {acc:.4f}  ({TP + TN} / {total})",
        fontsize=14, fontweight="bold", color="#1C2B3A", pad=18)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=200, bbox_inches="tight", facecolor="#F8FAFC")
    plt.close()


def plot_mrs_histogram(records: List[Dict],
                       mrs_overall: float,
                       mrs_pneu: float,
                       mrs_norm: float) -> None:
    all_ri  = np.array([r["Ri"] for r in records])
    ri_pneu = np.array([r["Ri"] for r in records if r["true_class"] == 1])
    ri_norm = np.array([r["Ri"] for r in records if r["true_class"] == 0])
    all_cov = np.array([r["lung_pixel_ratio"] for r in records])
    delta   = mrs_overall - MRS_BASELINE

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(all_ri, bins=30, color="steelblue",
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(mrs_overall,  color="tomato", lw=2,
                    label=f"MRS={mrs_overall:.4f}")
    axes[0].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--",
                    label=f"Baseline={MRS_BASELINE:.2f}")
    axes[0].set_xlabel("Ri"); axes[0].set_ylabel("Count")
    axes[0].set_title("MRS — All Samples")
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].hist(ri_pneu, bins=20, color="#FF69B4", edgecolor="white",
                 alpha=0.75, label=f"PNEUMONIA  {mrs_pneu:.4f}")
    axes[1].hist(ri_norm, bins=20, color="limegreen", edgecolor="white",
                 alpha=0.75, label=f"NORMAL  {mrs_norm:.4f}")
    axes[1].axvline(MRS_BASELINE, color="orange", lw=1.5, linestyle="--")
    axes[1].set_xlabel("Ri"); axes[1].set_ylabel("Count")
    axes[1].set_title("MRS — Per Class")
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
    fig.suptitle(
        f"MRS  |  ResNet-50 + Adversarial Training  |  "
        f"MRS={mrs_overall:.4f}  Δ={delta_str}  cov={all_cov.mean():.3f}",
        fontweight="bold", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mrs_histogram.png"),
                dpi=150, bbox_inches="tight")
    plt.close()


# ── EXPLANATION PANELS ────────────────────────────────────────────────────────

def _draw_panel(samples: List[Dict], class_label: str,
                accent: str, save_path: str) -> None:
    n = len(samples)
    if n == 0:
        return

    COL_TITLES = [
        "Original Image",
        "Adversarial Image\n(Pneumonia Highlighted)",
        "Adversarial Image\n(Pneumonia Heatmap)",
    ]

    fig = plt.figure(figsize=(13, 4.4 * n + 1.1))
    fig.patch.set_facecolor(BG)
    gs  = GridSpec(n + 1, 3, figure=fig,
                   height_ratios=[0.10] + [1.0] * n,
                   hspace=0.08, wspace=0.06)

    for ci, txt in enumerate(COL_TITLES):
        ax = fig.add_subplot(gs[0, ci])
        ax.set_facecolor(BG)
        ax.axhline(y=0.18, xmin=0.1, xmax=0.9, color=accent,
                   linewidth=3, solid_capstyle="round")
        ax.text(0.5, 0.65, txt, ha="center", va="center",
                color=TITLE_COL, fontsize=11, fontweight="bold",
                transform=ax.transAxes)
        ax.text(0.5, 0.28, class_label, ha="center", va="center",
                color=accent, fontsize=10, fontweight="bold",
                transform=ax.transAxes)
        ax.axis("off")

    for row, s in enumerate(samples):
        img     = s["img"]
        x_adv   = s["x_adv"]
        vis_sal = s["vis_sal"]
        prob    = s["prob"]
        pred    = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
        correct = (class_label == pred)
        ok_col  = "#2E7D32" if correct else "#C62828"

        ax0 = fig.add_subplot(gs[row + 1, 0])
        ax0.set_facecolor(CARD_BG)
        ax0.imshow(img, interpolation="bilinear")
        ax0.axis("off")
        ax0.set_title(f"GT: {class_label}  Pred: {pred}  p(PNEU): {prob:.3f}",
                      color=TITLE_COL, fontsize=7.5, pad=4)
        ax0.text(0.97, 0.97, "✓" if correct else "✗",
                 transform=ax0.transAxes, ha="right", va="top",
                 fontsize=14, fontweight="bold", color=ok_col,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white",
                           ec=ok_col, lw=1.5, alpha=0.85))

        ax1 = fig.add_subplot(gs[row + 1, 1])
        ax1.set_facecolor(CARD_BG)
        ax1.imshow(_adversarial_highlighted(x_adv, vis_sal),
                   interpolation="bilinear", aspect="auto")
        ax1.axis("off")
        ax1.set_title("Adversarial Image\n(Pneumonia Highlighted)",
                      color=SUB_COL, fontsize=7.5, pad=4)

        ax2 = fig.add_subplot(gs[row + 1, 2])
        ax2.set_facecolor(CARD_BG)
        im = ax2.imshow(_adversarial_heatmap(vis_sal),
                        interpolation="bilinear", aspect="auto",
                        vmin=0.0, vmax=1.0)
        ax2.axis("off")
        ax2.set_title("Adversarial Image\n(Pneumonia Heatmap)",
                      color=SUB_COL, fontsize=7.5, pad=4)
        _colorbar_on_ax(fig, ax2, im)

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + FGSM Adversarial Training"
        f"  |  ε={FGSM_EPS}  |  Colin & Surantha 2025",
        fontsize=12, fontweight="bold", color=TITLE_COL, y=1.002)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Panel saved  →  {save_path}")


def _render_single_trio(s: Dict, class_label: str,
                         accent: str, idx: int) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    fig.patch.set_facecolor(BG)
    for ax in axes:
        ax.set_facecolor(CARD_BG)

    img     = s["img"]
    x_adv   = s["x_adv"]
    vis_sal = s["vis_sal"]
    prob    = s["prob"]
    pred    = "PNEUMONIA" if prob >= 0.5 else "NORMAL"
    correct = (class_label == pred)
    ok_col  = "#2E7D32" if correct else "#C62828"
    sub     = f"GT: {class_label}   Pred: {pred}   p(PNEU): {prob:.3f}"

    axes[0].imshow(img, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title(f"Original Image\n{sub}", color=TITLE_COL, fontsize=9, pad=4)
    axes[0].text(0.97, 0.97, "✓" if correct else "✗",
                 transform=axes[0].transAxes, ha="right", va="top",
                 fontsize=15, fontweight="bold", color=ok_col,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white",
                           ec=ok_col, lw=1.5, alpha=0.85))

    axes[1].imshow(_adversarial_highlighted(x_adv, vis_sal),
                   interpolation="bilinear", aspect="auto")
    axes[1].axis("off")
    axes[1].set_title("Adversarial Image\n(Pneumonia Highlighted)",
                      color=SUB_COL, fontsize=9, pad=4)

    im = axes[2].imshow(_adversarial_heatmap(vis_sal),
                        interpolation="bilinear", aspect="auto",
                        vmin=0.0, vmax=1.0)
    axes[2].axis("off")
    axes[2].set_title("Adversarial Image\n(Pneumonia Heatmap)",
                      color=SUB_COL, fontsize=9, pad=4)
    cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.set_ticks([0.0, 0.5, 1.0])
    cbar.set_ticklabels(["Low", "Mid", "High"])
    cbar.ax.yaxis.set_tick_params(color=SUB_COL, labelsize=7)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUB_COL)
    cbar.outline.set_edgecolor("#CCCCCC")

    fig.suptitle(
        f"{class_label}  |  ResNet-50 + FGSM Adversarial Training"
        f"  |  Colin & Surantha 2025",
        color=TITLE_COL, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fname = f"single_{class_label.lower()}_{idx + 1:02d}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, fname),
                dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Single sample  →  {fname}")


# ── MRS COMPUTATION ───────────────────────────────────────────────────────────

def compute_mrs(test_gen, model: keras.Model,
                all_probs: np.ndarray,
                opt_thr: float,
                history: Dict) -> Tuple[Dict, List[Dict]]:
    print(f"\n{'='*60}\nCOMPUTING MRS\n{'='*60}")

    test_gen.reset()
    records:         List[Dict] = []
    panel_normal:    List[Dict] = []
    panel_pneumonia: List[Dict] = []
    sample_idx = 0
    n_failed   = 0
    N_PANEL    = 5

    for batch_x, batch_y in test_gen:
        B = len(batch_x)
        if sample_idx >= len(all_probs):
            break

        batch_end   = min(sample_idx + B, len(all_probs))
        batch_probs = all_probs[sample_idx:batch_end]
        actual_B    = len(batch_probs)
        batch_x     = batch_x[:actual_B]
        batch_y     = batch_y[:actual_B]

        pred_classes = (batch_probs >= opt_thr).astype(int)

        try:
            raw_sal_batch, x_adv_batch = compute_batch_saliency(
                model, batch_x, pred_classes)
        except Exception as exc:
            print(f"  WARNING batch at {sample_idx}: {exc}")
            n_failed += actual_B
            sample_idx += actual_B
            continue

        ri_arr, cov_arr = compute_ri_batch(batch_x, raw_sal_batch)

        for j in range(actual_B):
            true_class = int(batch_y[j])
            prob       = float(batch_probs[j])

            records.append(dict(
                sample_id        = sample_idx + j,
                true_class       = true_class,
                prob             = prob,
                pred_class       = int(pred_classes[j]),
                Ri               = float(ri_arr[j]),
                lung_pixel_ratio = float(cov_arr[j])))

            vis_sal = _postprocess_saliency(raw_sal_batch[j])
            entry   = dict(img=batch_x[j], x_adv=x_adv_batch[j],
                           vis_sal=vis_sal, prob=prob, true_class=true_class)
            if true_class == 0 and len(panel_normal)    < N_PANEL:
                panel_normal.append(entry)
            if true_class == 1 and len(panel_pneumonia) < N_PANEL:
                panel_pneumonia.append(entry)

        sample_idx += actual_B
        running = np.mean([r["Ri"] for r in records])
        ok_flag = ("✓" if LUNG_ADAPTIVE_LO <= cov_arr.mean()
                   <= LUNG_ADAPTIVE_HI else "⚠")
        print(f"  [{sample_idx:>4}/{test_gen.n}]  "
              f"MRS={running:.4f}  cov={cov_arr.mean():.2f}{ok_flag}")

    if not records:
        print("ERROR: No MRS records.")
        return {}, []

    all_ri  = np.array([r["Ri"] for r in records])
    ri_pneu = np.array([r["Ri"] for r in records if r["true_class"] == 1])
    ri_norm = np.array([r["Ri"] for r in records if r["true_class"] == 0])
    all_cov = np.array([r["lung_pixel_ratio"] for r in records])

    MRS_overall   = float(all_ri.mean())
    MRS_pneumonia = float(ri_pneu.mean()) if len(ri_pneu) else 0.0
    MRS_normal    = float(ri_norm.mean()) if len(ri_norm) else 0.0
    MRS_std       = float(all_ri.std())
    delta         = MRS_overall - MRS_BASELINE

    print(f"\n{'='*60}\nMEAN RELEVANCE SCORE\n{'='*60}")
    print(f"  MRS Overall   : {MRS_overall:.4f}  ±{MRS_std:.4f}")
    print(f"  MRS PNEUMONIA : {MRS_pneumonia:.4f}  (n={len(ri_pneu)})")
    print(f"  MRS NORMAL    : {MRS_normal:.4f}  (n={len(ri_norm)})")
    print(f"  Avg coverage  : {all_cov.mean():.3f}")
    print(f"  Baseline      : {MRS_BASELINE:.4f}")
    delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
    print(f"  Δ             : {delta_str}")
    if n_failed:
        print(f"  Failed        : {n_failed}")

    csv_path = os.path.join(OUTPUT_DIR, "mrs_per_sample.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader(); writer.writerows(records)

    plot_mrs_histogram(records, MRS_overall, MRS_pneumonia, MRS_normal)

    _draw_panel(panel_normal, "NORMAL", "limegreen",
                os.path.join(OUTPUT_DIR, "normal_panel.png"))
    _draw_panel(panel_pneumonia, "PNEUMONIA", "#FF69B4",
                os.path.join(OUTPUT_DIR, "pneumonia_panel.png"))

    for idx, s in enumerate(panel_normal[:5]):
        _render_single_trio(s, "NORMAL", "limegreen", idx)
    for idx, s in enumerate(panel_pneumonia[:5]):
        _render_single_trio(s, "PNEUMONIA", "#FF69B4", idx)

    if history.get("loss"):
        plot_training_combined(history, MRS_overall)

    mrs_results = dict(
        MRS_overall        = round(MRS_overall,   4),
        MRS_pneumonia      = round(MRS_pneumonia, 4),
        MRS_normal         = round(MRS_normal,    4),
        MRS_std            = round(MRS_std,        4),
        n_samples_mrs      = len(records),
        n_failed           = n_failed,
        baseline_colin2025 = MRS_BASELINE,
        delta_vs_baseline  = round(delta, 4),
        avg_lung_coverage  = round(float(all_cov.mean()), 4))

    return mrs_results, records


# ── ZIP ───────────────────────────────────────────────────────────────────────

def bundle_outputs() -> None:
    zip_path = os.path.join(OUTPUT_DIR, "resnet50_adversarial_fgsm_mrs.zip")
    exts = (".png", ".json", ".csv", ".keras")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fname in files:
                if any(fname.endswith(e) for e in exts):
                    full = os.path.join(root, fname)
                    zf.write(full, arcname=os.path.relpath(full, OUTPUT_DIR))
    print(f"ZIP  →  {zip_path}")


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main() -> None:
    print(f"ResNet-50 + FGSM Adversarial Training + MRS")
    print(f"EPOCHS={EPOCHS}  LR={LR}  L2={L2_VALUE}  ε={FGSM_EPS}  λ={LAMBDA_TRADEOFF}")
    print(f"Training objective: adversarial images only — Colin & Surantha 2025")

    train_gen, val_gen, test_gen, _, steps = get_loaders(DATA_ROOT)
    class_weight_dict = compute_class_weights_from_gen(train_gen)

    ckpt = os.path.join(OUTPUT_DIR, "best_model.keras")
    if os.path.exists(ckpt):
        print(f"Loading checkpoint  ←  {ckpt}")
        model = keras.models.load_model(ckpt)
        hist_path = os.path.join(OUTPUT_DIR, "history.json")
        history = (json.load(open(hist_path)) if os.path.exists(hist_path)
                   else {"loss": [], "accuracy": [], "auc": [],
                         "val_loss": [], "val_accuracy": [], "val_auc": [],
                         "trade_off_score": []})
    else:
        model = build_resnet50()
        model.summary(line_length=90)
        model, history = train_model_adversarial(
            model, train_gen, val_gen, steps, class_weight_dict)
        with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as fh:
            json.dump(history, fh, indent=2)

    if history.get("loss"):
        plot_training_accuracy(history)
        plot_training_loss(history)

    metrics, opt_thr, all_labels, all_probs = evaluate_model(
        model, val_gen, test_gen)

    plot_confusion_matrix(metrics)
    plot_ppv_npv(metrics)

    test_gen.reset()
    mrs_results, records = compute_mrs(
        test_gen, model, all_probs, opt_thr, history)

    if mrs_results:
        plot_bar_chart_metrics(metrics, mrs_results["MRS_overall"])

    trade_off = (metrics["accuracy"]
                 - LAMBDA_TRADEOFF
                 * (1.0 - mrs_results.get("MRS_overall", 0.0)))

    avg_cov = mrs_results.get("avg_lung_coverage", 0.0)
    mrs_val = mrs_results.get("MRS_overall", 0.0)
    print(f"\n{'─'*50}")
    if   avg_cov < LUNG_ADAPTIVE_LO:
        print(f"⚠  cov={avg_cov:.3f} < floor — decrease pad constants")
    elif avg_cov > LUNG_ADAPTIVE_HI:
        print(f"⚠  cov={avg_cov:.3f} > ceil  — increase pad constants")
    else:
        print(f"✓  cov={avg_cov:.3f}  lung mask OK")
    if   mrs_val >= MRS_BASELINE:
        print(f"✓  MRS={mrs_val:.4f} ≥ {MRS_BASELINE}  TARGET ACHIEVED")
    elif mrs_val >= 0.70:
        print(f"△  MRS={mrs_val:.4f}  — check lung mask coverage")
    else:
        print(f"✗  MRS={mrs_val:.4f}  — check model quality")
    print(f"   Trade-off score (λ={LAMBDA_TRADEOFF}): {trade_off:.4f}")

    final = {
        **{k: v for k, v in metrics.items()
           if k not in ("TP", "TN", "FP", "FN")},
        **mrs_results,
        "trade_off_score"           : round(trade_off, 4),
        "lambda"                    : LAMBDA_TRADEOFF,
        "interpretability_technique": "FGSM Adversarial Training",
        "training_objective"        : "adversarial images only (paper-accurate)",
        "fgsm_epsilon"              : FGSM_EPS,
        "fgsm_lr"                   : FGSM_LR,
        "saliency_method"           : "gradient saliency |∂L/∂x|",
        "visualization_col1"        : "Original Image",
        "visualization_col2"        : "Adversarial Image (Pneumonia Highlighted)",
        "visualization_col3"        : "Adversarial Image (Pneumonia Heatmap)",
        "mrs_formula"               : "Ri = Σ(saliency_pos·M)/(Σ saliency_pos+ε)",
        "reference"                 : "Colin & Surantha 2025  |  Goodfellow et al. 2015",
        "threshold_source"          : "validation set only",
        "epochs": EPOCHS, "lr": LR, "l2": L2_VALUE}

    with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w") as f:
        json.dump({k: round(float(v), 4) if isinstance(v, float) else v
                   for k, v in final.items()}, f, indent=2)

    bundle_outputs()
    print(f"\nAll outputs  →  {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

2026-05-19 03:03:38.026535: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779159818.225063      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779159818.289420      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779159818.754647      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779159818.754686      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779159818.754689      57 computation_placer.cc:177] computation placer alr

TF 2.19.0 | GPUs: 2
ResNet-50 + FGSM Adversarial Training + MRS
EPOCHS=50  LR=0.001  L2=0.0001  ε=0.01  λ=0.5
Training objective: adversarial images only — Colin & Surantha 2025
Found 4708 validated image filenames belonging to 2 classes.
Found 524 validated image filenames belonging to 2 classes.
Found 624 validated image filenames belonging to 2 classes.
Train=4708  Val=524  Test=624
Class map: {'NORMAL': 0, 'PNEUMONIA': 1}
Class weights: NORMAL=1.939  PNEUMONIA=0.674


I0000 00:00:1779159863.098434      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779159863.104501      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "ResNet50_Pneumonia"

...


  ResNet-50 + FGSM Adversarial Training  |  ε=0.01  LR=0.001
  Training objective: adversarial images only (Colin & Surantha 2025)



I0000 00:00:1779159899.118519     129 cuda_dnn.cc:529] Loaded cuDNN version 91002


Epoch   1/50  loss=0.1442  acc=0.9839  val_loss=152599.5106  val_acc=0.2576  tradeoff=0.1376
  ✓ val_loss improved: inf → 152599.5106  [model saved]
Epoch   2/50  loss=0.1709  acc=0.9673  val_loss=2.2065  val_acc=0.7424  tradeoff=0.6224
  ✓ val_loss improved: 152599.5106 → 2.2065  [model saved]
Epoch   3/50  loss=0.2180  acc=0.9157  val_loss=2.5134  val_acc=0.7424  tradeoff=0.6224
  ✗ val_loss did not improve (2.5134 ≥ 2.2065)  patience 1/10
Epoch   4/50  loss=0.1673  acc=0.9378  val_loss=7.9334  val_acc=0.7424  tradeoff=0.6224
  ✗ val_loss did not improve (7.9334 ≥ 2.2065)  patience 2/10
Epoch   5/50  loss=0.0359  acc=0.9924  val_loss=15.8458  val_acc=0.7424  tradeoff=0.6224
  ✗ val_loss did not improve (15.8458 ≥ 2.2065)  patience 3/10
Epoch   6/50  loss=0.2257  acc=0.9102  val_loss=0.7020  val_acc=0.2576  tradeoff=0.1376
  ✓ val_loss improved: 2.2065 → 0.7020  [model saved]
Epoch   7/50  loss=0.2941  acc=0.8664  val_loss=1.9833  val_acc=0.7443  tradeoff=0.6243
  ✗ val_loss did not i

I0000 00:00:1779171069.306691     129 service.cc:152] XLA service 0x78abbf856bb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779171069.306773     129 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779171069.306782     129 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779171074.063639     129 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Validation threshold (sensitivity ≥ 0.90): 0.9432

TEST RESULTS
  Threshold (val)   : 0.9432
  Accuracy          : 0.9279
  Sensitivity       : 0.9462
  Specificity       : 0.8974
  PPV               : 0.9389
  NPV               : 0.9091
  F1                : 0.9425
  AUC-ROC           : 0.9745  95% CI [0.9620, 0.9846]

COMPUTING MRS
  [  32/624]  MRS=0.8378  cov=0.78✓
  [  64/624]  MRS=0.8350  cov=0.78✓
  [  96/624]  MRS=0.8324  cov=0.78✓
  [ 128/624]  MRS=0.8337  cov=0.78✓
  [ 160/624]  MRS=0.8327  cov=0.78✓
  [ 192/624]  MRS=0.8306  cov=0.78✓
  [ 224/624]  MRS=0.8320  cov=0.78✓
  [ 256/624]  MRS=0.8318  cov=0.76✓
  [ 288/624]  MRS=0.8300  cov=0.75✓
  [ 320/624]  MRS=0.8293  cov=0.77✓
  [ 352/624]  MRS=0.8257  cov=0.75✓
  [ 384/624]  MRS=0.8246  cov=0.77✓
  [ 416/624]  MRS=0.8233  cov=0.76✓
  [ 448/624]  MRS=0.8233  cov=0.76✓
  [ 480/624]  MRS=0.8233  cov=0.76✓
  [ 512/624]  MRS=0.8238  cov=0.76✓
  [ 544/624]  MRS=0.8237  cov=0.76✓
  [ 576/624]  MRS=0.8233  cov=0.76✓
  [ 608/624]  MR